In [2]:
from __future__ import annotations

import os
import sys

try:
    import anndata
    import scanpy
except ImportError:
    print("Instalando dependências compatíveis com o ambiente do Colab...")
    # Mantém o pandas travado na versão esperada pelo Colab (2.2.3)
    !pip install -q "pandas==2.2.3" anndata scanpy

Instalando dependências compatíveis com o ambiente do Colab...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 145.6 MB/s eta 0:00:0000:01


In [ ]:
REPO_NAME = "pipiline_hopifield"
REPO_URL = "https://github.com/letdevx/pipiline_hopifield.git"
DEST_PATH = f"/content/{REPO_NAME}"

# Clona ou atualiza o repositório na VM do Colab
if os.path.exists("/content"):
    if not os.path.exists(DEST_PATH):
        print("Clonando código para a VM...")
        os.system(f"git clone {REPO_URL} {DEST_PATH}")
    else:
        print("Atualizando código na VM...")
        os.system(f"cd {DEST_PATH} && git pull")

    os.system(f"cd {DEST_PATH} && git checkout teste-pipeline_genereico_Pan_F")

# Adiciona a raiz do repo e a pasta 'src' ao sys.path
for _p in (DEST_PATH, os.path.join(DEST_PATH, "src")):
    if os.path.exists(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

In [ ]:
try:
    from google.colab import drive  # type: ignore

    if not os.path.exists("/content/drive"):
        print("[Colab] Montando Google Drive em /content/drive...")
        drive.mount("/content/drive")
except (ImportError, Exception):
    pass

In [ ]:
import gc
import os
import sys
from pathlib import Path
import anndata as ad
import numpy as np
import polars as pl
import scipy.io as sio
import scipy.sparse as sp

: 

In [9]:
# Resolução dinâmica e robusta do diretório raiz e de src/ (Colab e Local)
for _raiz in [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/pipiline_hopifield"),
    Path("/content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/pipiline_hopifield"),
]:
    if (_raiz / "src").is_dir() and str(_raiz) not in sys.path:
        sys.path.insert(0, str(_raiz))
        sys.path.insert(0, str(_raiz / "src"))
        break

if os.path.exists("/content") and not os.path.exists("/content/pipiline_hopifield"):
    print("Clonando repositório na VM do Colab para carregar src...")
    os.system("git clone -b teste-pipeline_genereico_Pan_F https://github.com/letdevx/pipiline_hopifield.git /content/pipiline_hopifield")
    for _p in ("/content/pipiline_hopifield", "/content/pipiline_hopifield/src"):
        if _p not in sys.path:
            sys.path.insert(0, _p)

from src.config import OUTPUTS, PATH_BASE, PATH_ORTHBASE_RDS
from src.treinamento import ProjetorSWeePR

# 1. Definição e criação do diretório usando Pathlib
DIR_PROJECAO_AUSENTES = Path(OUTPUTS) / "projecao_ausentes_pan"
DIR_PROJECAO_AUSENTES.mkdir(parents=True, exist_ok=True)

# 2. Caminhos dos arquivos de entrada e saída
PATH_MTX_ENTRADA = DIR_PROJECAO_AUSENTES / "matrix.mtx"
PATH_SAIDA_TXT = DIR_PROJECAO_AUSENTES / "matriz_sweep_ausentes.txt"
PATH_SAIDA_NPY = DIR_PROJECAO_AUSENTES / "matriz_sweep_ausentes.npy"

print(f"Diretório de saída pronto: {DIR_PROJECAO_AUSENTES}")
print(f"OrthBase canônica configurada: {PATH_ORTHBASE_RDS}")

SyntaxError: Expected one or more names after 'import' (2141934000.py, line 1)

: 

In [7]:
# Resolução dinâmica dos arquivos de entrada (Colab Google Drive com fallback para local PATH_BASE)
caminho_tracking_colab = (
    r"/content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop"
    r"/outputs/alinhamento/tracking_genes_adicionados_mathys.csv"
)
caminho_tracking_local = os.path.join(
    PATH_BASE, "outputs", "alinhamento", "tracking_genes_adicionados_mathys.csv"
)
tracking_pan_F = (
    caminho_tracking_colab
    if os.path.exists(caminho_tracking_colab)
    else caminho_tracking_local
)

caminho_pan_colab = (
    r"/content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop"
    r"/imputs/PAN_Bin/pan_anotado.h5ad"
)
caminho_pan_local = os.path.join(PATH_BASE, "imputs", "pan_anotado.h5ad")
matriz_pan = (
    caminho_pan_colab if os.path.exists(caminho_pan_colab) else caminho_pan_local
)

print(f"Tracking CSV : {tracking_pan_F} (Existe: {os.path.exists(tracking_pan_F)})")
print(f"Matriz Pan   : {matriz_pan} (Existe: {os.path.exists(matriz_pan)})")

: 

In [ ]:
posicao_genes_none_p_f = pl.read_csv(tracking_pan_F)
posicao_genes_none_p_f.head(5)

gene_name,ensembl_id,posicao_coluna,valor_inserido,presente_fujita,presente_mathys
str,str,i64,f64,bool,bool
"""MTND1P23""","""ENSG00000225972""",5,0.5,true,false
"""RPL7P7""","""ENSG00000224315""",6,0.5,true,false
"""MTCO3P12""","""ENSG00000198744""",7,0.5,true,false
"""DDX11L17""","""ENSG00000279928""",8,0.5,true,false
"""MTND2P28""","""ENSG00000225630""",11,0.5,true,false


: 

In [ ]:
posicao_coluna_pan = posicao_genes_none_p_f["posicao_coluna"]
print(f"Total de genes ausentes mapeados: {posicao_coluna_pan.shape[0]}")

(25065,)

: 

In [ ]:
indice_pan = posicao_coluna_pan.to_list()
print(f"Primeiros 10 índices de colunas ausentes: {indice_pan[:10]}")

[5, 6, 7, 8, 11, 18, 19, 21, 25, 28, 29, 30, 33, 37, 39, 40, 46, 49, 50, 53, 55, 57, 60, 61, 72, 74, 77, 79, 83, 89, 93, 97, 102, 104, 105, 115, 128, 134, 138, 140, 143, 151, 156, 168, 170, 172, 175, 179, 180, 185, 191, 198, 199, 200, 201, 207, 210, 218, 221, 223, 228, 231, 238, 240, 242, 244, 246, 258, 259, 265, 268, 270, 281, 285, 287, 289, 293, 294, 303, 305, 309, 312, 313, 318, 321, 326, 329, 333, 335, 338, 345, 349, 354, 358, 359, 361, 364, 367, 369, 372, 380, 382, 386, 388, 392, 397, 404, 407, 411, 417, 419, 421, 422, 424, 425, 427, 429, 437, 438, 444, 446, 449, 451, 456, 458, 463, 467, 470, 472, 475, 478, 485, 487, 491, 492, 500, 510, 512, 513, 515, 516, 524, 525, 526, 527, 528, 534, 537, 543, 544, 546, 549, 558, 559, 560, 568, 570, 575, 580, 583, 584, 585, 586, 590, 592, 597, 599, 600, 601, 603, 613, 614, 615, 620, 622, 625, 628, 637, 640, 641, 643, 644, 647, 649, 652, 657, 659, 662, 666, 667, 671, 677, 684, 686, 694, 697, 699, 703, 705, 707, 708, 709, 712, 714, 718, 719, 720, 

: 

### Injeção de Sentinela Neutro (0.5) nas Colunas Ausentes do Pan
Injeta o valor sentinela neutro 0.5 (canônico do pipeline Hopfield) nas colunas de genes ausentes.
A operação preserva o formato esparso da matriz AnnData para economizar memória RAM e evitar OOM.

In [ ]:
print(f"Carregando matriz AnnData: {matriz_pan}...")
adata = ad.read_h5ad(matriz_pan)

# Garante manipulação eficiente de memória com LIL/CSR sem conversão densa
if sp.issparse(adata.X):
    X_mod = adata.X.tolil()
    X_mod[:, indice_pan] = 0.5
    X_mod = X_mod.tocsr()
else:
    X_mod = np.asarray(adata.X, dtype=np.float32).copy()
    X_mod[:, indice_pan] = 0.5
    X_mod = sp.csr_matrix(X_mod)

print(f"Matriz modificada: {X_mod.shape[0]} células × {X_mod.shape[1]} genes (nnz: {X_mod.nnz})")

### Exportação Direta no Formato Matrix Market (.mtx)
Salva a matriz esparsa modificada em disco via `scipy.io.mmwrite` de forma rápida e enxuta.

In [ ]:
print(f"Exportando matriz para formato Matrix Market (.mtx): {PATH_MTX_ENTRADA}...")
sio.mmwrite(str(PATH_MTX_ENTRADA), X_mod)
print(f"Exportação MTX concluída com sucesso: {PATH_MTX_ENTRADA.stat().st_size / 1e6:.2f} MB")

# Liberação preventiva de memória RAM
del adata, X_mod
gc.collect()

: 

### Projeção rSWeeP Canônica com Reuso da Base Congelada Padrão
Executa a projeção rSWeeP oficial reutilizando a base ortonormal canônica congelada (`PATH_ORTHBASE_RDS`).
O resultado compactado (600 dimensões) é persistido em `.txt` e `.npy`.

In [ ]:
# Garante que dependências R estejam disponíveis caso executado no Colab
ProjetorSWeePR.verificar_e_instalar_dependencias_r()

[[0.  0.  0.  0.  0.  0.5 0.5 0.5 0.5 0. ]
 [0.  0.  0.  0.  0.  0.5 0.5 0.5 0.5 0. ]
 [0.  0.  1.  0.  0.  0.5 0.5 0.5 0.5 0. ]
 [0.  0.  0.  0.  0.  0.5 0.5 0.5 0.5 0. ]
 [0.  0.  0.  0.  0.  0.5 0.5 0.5 0.5 0. ]
 [0.  0.  0.  0.  0.  0.5 0.5 0.5 0.5 0. ]
 [0.  0.  0.  0.  0.  0.5 0.5 0.5 0.5 0. ]
 [0.  0.  1.  0.  0.  0.5 0.5 0.5 0.5 0. ]
 [0.  0.  0.  0.  0.  0.5 0.5 0.5 0.5 0. ]
 [0.  0.  1.  0.  0.  0.5 0.5 0.5 0.5 0. ]]


: 

In [ ]:
print(f"[rSWeeP] Inicializando projetor oficial para {PATH_MTX_ENTRADA}...")
projetor = ProjetorSWeePR(
    path_matriz=str(PATH_MTX_ENTRADA),
    path_saida=str(PATH_SAIDA_TXT),
    n_componentes=600,
    seed=42,
    path_orthbase=str(PATH_ORTHBASE_RDS),
)

# Dispara o subprocesso oficial em R (orthBase + SWeeP)
projetor.projetar()

# Validações de integridade pós-projeção
assert projetor.Wswp is not None, "Erro: Matriz projetada retornou nula!"
assert not np.isnan(projetor.Wswp).any(), "Erro: Detectados valores NaN na projeção!"

# Persistência em formato binário NumPy (.npy)
np.save(str(PATH_SAIDA_NPY), projetor.Wswp)

print("\n=======================================================")
print(" [rSWeeP] Projeção de Ausentes Pan Concluída com Sucesso!")
print(f"  Shape final : {projetor.Wswp.shape} (células × 600 dimensões)")
print(f"  TXT salvo em: {PATH_SAIDA_TXT}")
print(f"  NPY salvo em: {PATH_SAIDA_NPY}")
print(f"  OrthBase    : {projetor.path_orthbase}")
print("=======================================================")